# 01 — REF concepts

This notebook introduces the core vocabulary of the **Rapid Evaluation Framework (REF)**.
By the end you will know what a *diagnostic*, *provider*, *execution*, *metric* and *dataset* are, and how they fit together.

**Prerequisites:** none. This is the place to start.

**What you need:** an internet connection as we read from the public REF API.

## What is the REF?

The REF evaluates climate datasets as new model output becomes available and produces
figures and metrics. The public deployment now includes CMIP7 Assessment Fast Track
results alongside CMIP6 results. These notebooks use CMIP7 examples.

The [REF dashboard](https://dashboard.climate-ref.org) offers a curated view of the results.
The [public API](https://api.climate-ref.org) provides the same results in a structured form,
and its [schema](https://api.climate-ref.org/docs) describes what can be queried.

Throughout these notebooks we talk to the API via an "SDK" which is a small software package generated from the schema of the API.
This SDK allows us to make requests to the API as function calls without having to directly make HTTP requests.

A small script `scripts/generate_client.sh` shows how to built the SDK.

> If you are runnning this on Binder, this has been pre-built for you.

In [1]:
from ref_tutorials import get_client

client = get_client()
client

Client(raise_on_unexpected_status=False, _base_url='https://api.climate-ref.org', _cookies={}, _headers={}, _timeout=None, _verify_ssl=True, _follow_redirects=False, _httpx_args={}, _client=None, _async_client=None)

## Datasets

Datasets containing CMORised climate model output, and the reference data it is compared to are a key input to the REF.
The REF tracks which datasets are available and which diagnostics have already been run.

## Diagnostics

A **diagnostic** is a single, well-defined evaluation to understand a component of the earth system.
For example "the global mean surface temperature timeseries" or "the Atlantic overturning circulation strength".

Let's list the diagnostics the REF currently provides:

In [2]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
print(f"{len(diagnostics)} diagnostics available. Showing the first 10:\n")
for diagnostic in sorted(diagnostics, key=lambda d: d.name)[:10]:
    print(f"  {diagnostic.slug:35s} {diagnostic.name}")

47 diagnostics available. Showing the first 10:

  annual-cycle                        Annual Cycle Analysis
  sea-ice-area-basic                  Arctic and Antarctic Sea Ice Area Seasonal Cycle
  amoc-rapid                          Atlantic Meridional Overturning Circulation (RAPID)
  burntfractionall-gfed               Burnt Fraction (GFED)
  climate-drivers-for-fire            Climate Drivers for Fire
  climate-at-global-warming-levels    Climate at Global Warming Levels
  cloud-radiative-effects             Cloud Radiative Effects
  cloud-scatterplots-reference        Cloud Scatterplots for Reference dataset
  cloud-scatterplots-clwvi-pr         Cloud-Precipitation Scatterplots (clwvi vs pr)
  cloud-scatterplots-clivi-lwcre      Cloud-Radiation Scatterplots (clivi vs lwcre)


Every diagnostic has a human-readable `name`, a short machine-friendly `slug`, and a
`description`. We will use the global-mean temperature timeseries diagnostic, which has
CMIP7 results:

In [3]:
example = next(
    d for d in diagnostics if d.provider.slug == "esmvaltool" and d.slug == "global-mean-timeseries"
)
print("name:       ", example.name)
print("slug:       ", example.slug)
print("description:", example.description.strip())

name:        Global Mean Timeseries
slug:        global-mean-timeseries
description: Plot the global mean time series of near-surface air temperature (tas). Example diagnostic with no observational datasets.


## Providers

Diagnostics are grouped into **providers**. A **provider** is a package that knows how to compute a family of diagnostics.

For the Assessment Fast Track we use:

- ESMValTool
- PCMDI Metrics Package (PMP)
- ILAMB/IOMB

The REF orchestrates providers and does not compute diagnostics itself.
Each provider is a thin wrapper around the upstream diagnostics package.

Each diagnostic tells you which provider it belongs to:

In [4]:
providers = {d.provider.slug: d.provider.name for d in diagnostics}
for slug, name in sorted(providers.items()):
    count = sum(1 for d in diagnostics if d.provider.slug == slug)
    print(f"  {slug:20s} {name}  ({count} diagnostics)")

  esmvaltool           ESMValTool  (24 diagnostics)
  ilamb                ILAMB  (13 diagnostics)
  pmp                  PMP  (10 diagnostics)


## Execution groups and executions

An **execution** is one run of a diagnostic against a particular set of datasets.
It records which data were used and what the run produced.

A diagnostic can run for many models and experiments. Each distinct selection is an
*execution group*. Its key describes the selection. If its inputs change, REF can run
it again, so one group may contain several executions.

We list the CMIP7 execution groups for the global-mean timeseries diagnostic:

In [5]:
from climate_ref_client.api.diagnostics import diagnostics_list_execution_groups

from ref_tutorials import latest_successful_execution

execution_groups = diagnostics_list_execution_groups.sync(
    client=client,
    provider_slug=example.provider.slug,
    diagnostic_slug=example.slug,
    mip_era="CMIP7",
).data

first_execution_group = next(
    (group for group in execution_groups if latest_successful_execution(group) is not None),
    None,
)

print(f"Diagnostic: {example.name}")
print(f"CMIP7 execution groups: {len(execution_groups)}")
print(f"First five keys: {[group.key for group in execution_groups[:5]]}")

Diagnostic: Global Mean Timeseries
CMIP7 execution groups: 496
First five keys: ['cmip7_piControl_g120_CanESM5-1_tas_r1i1p2f1', 'cmip7_1pctCO2_g110_UKCM2-0-LL_tas_r1i1p1f1', 'cmip7_historical_g137_UKCM2a-0-HH_tas_r1i1p1f1', 'cmip7_esm-hist_g110_UKESM1-3-LL_tas_r1i1p1f1', 'cmip7_esm-piControl_g110_UKESM1-3-LL_tas_r1i1p1f1']


Within each execution group, there may be multiple executions. We pick the most recent
successful one and inspect the files it produced. Each output has a URL for downloading it.

In [6]:
execution = latest_successful_execution(first_execution_group) if first_execution_group else None
if execution is None:
    print("No successful CMIP7 execution is available yet.")
else:
    print(f"Execution group: {first_execution_group.key}")
    for output in execution.outputs:
        print(f"  {output.filename:20s} {output.url}")

Execution group: cmip7_piControl_g120_CanESM5-1_tas_r1i1p2f1
  executions/recipe_20260904_151507/plots/timeseries/script1/png/tas_tavg-h2m-hxy-u_mon_glb_g120_CanESM5-1_piControl_r1i1p2f1_65000116T120000-69001216T120000.png https://api.climate-ref.org/api/v1/results/14658
  executions/recipe_20260904_151507/work/timeseries/script1/tas_tavg-h2m-hxy-u_mon_glb_g120_CanESM5-1_piControl_r1i1p2f1_65000116T120000-69001216T120000.nc https://api.climate-ref.org/api/v1/results/14659
  executions/recipe_20260904_151507/index.html https://api.climate-ref.org/api/v1/results/14660


## Metrics

An execution produces output. That output comes in two main shapes:

- **Metric values** — numbers that summarise a property of a model. 
  A *scalar* is a single number (e.g. a root-mean-square error) or a *series* is a number per index point
  (e.g. a seasonal cycle).
- **Files** — NetCDF data and figures for deeper analysis or custom plotting.

The REF ingests all of these outputs into its database so they can be queried by the API.

Notebook 02 shows how to retrieve metric values, and notebook 03 turns them into a figure.

## Recap

| Term                | Meaning                                                                       |
| ------------------- | ----------------------------------------------------------------------------- |
| **Provider**        | A package that computes a family of diagnostics (ESMValTool, PMP, ILAMB, ...) |
| **Diagnostic**      | A single well-defined evaluation                                              |
| **Dataset**         | Climate model output or reference data a diagnostic runs against              |
| **Execution**       | One run of a diagnostic against one group of datasets                         |
| **Metric value**    | A scalar or series result summarising a model                                 |

**Next:** [02 — Querying the REF API](02-querying-the-api.ipynb).